# Proyecto UCU

In [ ]:
import shutil
import os
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## Python para Análisis de Datos

#### Análisis de canciones de Spotify

## 1) Definición del Problema

Contamos con un dataset proveniente de Kaggle que se obtuvo utilizando la API de Spotify. Nuestro objetivo será procesarlo para obtener un dataset adecuado con el que se puedan visualizar de forma fácil las relaciones entre los atributos de las canciones.

## Paso 2: Recopilación de datos


Descarga del dataset con kagglehub y guardado en la carpeta _../data/raw_.

### Importación de los datos y creación del DataFrame


In [ ]:
path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")
os.makedirs("../data/raw", exist_ok=True)
shutil.copytree(path, "../data/raw", dirs_exist_ok=True)

display("Spotify Tracks Dataset descargado en: ../data/raw")

In [ ]:
df = pd.read_csv("../data/raw/dataset.csv")


Exploración de los campos del dataset:

|     Variable     | Definición                                                                                                                       | Key                                                                                                                |
|:----------------:|:---------------------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------|
|   Unnamed: 0   | Índice del registro                                                                                                              |                                                                                                                    |
|     track_id     | Identificador único de la canción                                                                                                |                                                                                                                    |
|     artists      | Intérprete/s                                                                                                                     |                                                                                                                    |
|    album_name    | Nombre del álbum                                                                                                                 |                                                                                                                    |
|    track_name    | Nombre de la canción                                                                                                             |                                                                                                                    |
|    popularity    | Popularidad de la canción                                                                                                        | 0 = Menor popularidad, 100 = Mayor popularidad                                                                     |
|   duration_ms    | Duración de la canción en milisegundos                                                                                           |                                                                                                                    |
|     explicit     | Indica si la canción contiene contenido explícito                                                                                | False = No, True = Sí                                                                                              |
|   danceability   | Bailabilidad (o cualidad de bailable)                                                                                            | 0 = Menos bailable, 1 = Más bailable                                                                               |
|      energy      | Medida de la intensidad de la canción                                                                                            | 0 = Baja energía, 1 = Alta energía                                                                                 |
|       key        | Tonalidad en la que está la canción                                                                                              | 0 = C, 1 = C♯/D♭, 2 = D, 3 = D♯/E♭, 4 = E, 5 = F, 6 = F♯/G♭, 7 = G, 8 = G♯/A♭, 9 = A, 10 = A♯/B♭, 11 = B |
|     loudness     | Volumen de la canción en decibeles (dB)                                                                                          | Valores relativos al máximo digital (0dB)                                                                 |
|       mode       | Indica si la tonalidad de la canción está en modo mayor o menor                                                                  | 0 = Menor, 1 = Mayor                                                                                               |
|   speechiness    | Presencia de palabras habladas en la canción                                                                                     | 0 = Poco contenido hablado, 1 = Mucho contenido hablado                                                            |
|   acousticness   | Métrica musical que mide el nivel de confianza de que la canción utilice sonidos en vivo en lugar de procesados electrónicamente | 0 = Baja probabilidad, 1 = Alta probabilidad                                                                       |
| instrumentalness | Predice si una canción no contiene voces                                                                                         | 0 = Contiene voces, 1 = Instrumental                                                                               |
|     liveness     | Probabilidad de que la grabación haya sido realizada en vivo                                                                     | 0 = Baja probabilidad, 1 = Alta probabilidad                                                                       |
|     valence      | Medida que describe la positividad musical de una canción                                                                        | 0 = Más negativa/triste, 1 = Más positiva/alegre                                                                   |
|      tempo       | Tempo estimado de la canción en pulsaciones o beats por minuto (BPM)                                                             |                                                                                                                    |
|  time_signature  | Compás estimado de la canción                                                                                                    | 3 = 3/4, 4 = 4/4, 5 = 5/4                                                                                          |
|   track_genre    | Género musical de la canción                                                                                                     |                                                                                                                    |

El dataset contiene varios valores que provienen de modelos predictivos de Spotify, por lo que es posible que existan errores y es algo a tener en cuenta en el posterior análisis.

## Paso 3: Análisis Descriptivo

In [ ]:
df

* Se observa que en la columna track_name hay 89741 valores únicos en un dataset con 114000 registros, pero no se trata de un error. Para una misma canción el dataset tiene una fila por cada género de la pista.

In [ ]:
df.info()

* 3 de las 21 columnas presentan un único dato null.
* El data type detectado para cada columna concuerda con el tipo de dato esperado en todos los casos, pero para las variables key, mode y time_signature podrían redefinirse como variables categóricas.
* La columna "Unnamed: 0" funciona como identificador.

In [ ]:
df.describe().T

* Se detectan algunas incongruencias como los mínimos iguales a 0.000 para duration_ms y tempo.
* Se procede a identificar la cantidad de valores "0" en cada columna

In [ ]:
(df == 0).sum().sort_values(ascending=False)

* En algunas de estas columnas, los valores iguales a "0" están dentro de lo esperado. Por ejemplo para mode que es una variable booleana, un valor de cero corresponde a una canción en tonalidad menor. Sin embargo, esto no tiene sentido para las columnas valence, tempo y danceability, por lo que es algo a considerar al utilizar estos campos para relacionarlos con el resto.
* Procedemos a buscar filas con datos nulos:

In [ ]:
df.isnull().sum()

In [ ]:
df[df.isnull().any(axis=1)]

* Los tres nulos en las columnas artists, album_name y track_name corresponden a la fila con id 65900.

* Continuamos buscando duplicados:

In [ ]:
df.duplicated()

* No se encontraron duplicados. Podemos extender la búsqueda filtrando la columna track_id para evaluar si hay dos canciones con diferente track_id pero igual contenido (la misma canción con los mismos valores pero asignada a dos track_id diferentes):

In [ ]:

df.duplicated(subset=df.columns.difference(['track_id'])).sum()

* Ninguna canción aparece más de una vez con diferente track_id.

## Paso 4: Limpieza de Datos



* Se eliminaron las 176 filas que presentaban un valor igual a 0 en al menos una de las variables tempo, danceability o valence ya que un valor cero en alguna de estas variables puede dificultar o distorsionar la comparación entre canciones.
* También se decidió eliminar la fila que contenia datos nulos por el mismo motivo.

In [ ]:
df_clean = df.copy()

In [ ]:
filas_cero = (
    (df_clean["tempo"] == 0) |
    (df_clean["danceability"] == 0) |
    (df_clean["valence"] == 0)
)

filas_cero.sum()

In [ ]:
df_clean.drop(df_clean[filas_cero].index, inplace=True)


In [ ]:
df_clean.drop(65900, axis=0, inplace=True)

## Paso 5: Análisis de Variables

## Paso 6: Ingeniería de características
